In [1]:
import pandas as pd
import numpy as np
import os
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, cohen_kappa_score
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV




In [2]:
for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

train_df1 = pd.read_csv(
    "/kaggle/input/learning-agency-lab-automated-essay-scoring-2/train.csv"
)




/kaggle/input/description.md
/kaggle/input/sample_submission.csv
/kaggle/input/sample_submission.csv.zip
/kaggle/input/test.csv
/kaggle/input/train.csv.zip
/kaggle/input/train.csv
/kaggle/input/test.csv.zip
/kaggle/input/learning-agency-lab-automated-essay-scoring-2/description.md
/kaggle/input/learning-agency-lab-automated-essay-scoring-2/sample_submission.csv
/kaggle/input/learning-agency-lab-automated-essay-scoring-2/sample_submission.csv.zip
/kaggle/input/learning-agency-lab-automated-essay-scoring-2/test.csv
/kaggle/input/learning-agency-lab-automated-essay-scoring-2/train.csv.zip
/kaggle/input/learning-agency-lab-automated-essay-scoring-2/train.csv
/kaggle/input/learning-agency-lab-automated-essay-scoring-2/test.csv.zip


In [3]:
train_df1["score"].dtypes




dtype('int64')

In [4]:
train_df1.head(10)




,essay_id,full_text,score
0,663d2cf,"Dear State Senator,\n\nI am arguing in favor o...",3
1,3a20bfb,"In "" The Challenge of Exploring Venus"" The aut...",2
2,6adae64,Teachers can have a hard time telling if their...,3
3,c81ccdf,Using dirverless cars can be very dangrous in ...,3
4,9549d7f,"In "" The Challenge of Exploring Venus"" the aut...",3
5,15992f4,What would the world be like if we could just ...,3
6,1b7e42c,Alien Landform?\n\nDo you think that the face ...,1
7,e06d5d9,"My argument is for ""Driverless cars"", the reas...",3
8,a779bc6,Have you ever seen those movies will the smart...,4
9,ca7b3c6,"Dont you get tired of driving sometimes, Get s...",2


In [5]:
train_df1["score"].value_counts()




score
3    5629
2    4249
4    3563
1    1124
5     876
6     135
Name: count, dtype: int64

In [6]:
submission = pd.read_csv(
    "/kaggle/input/learning-agency-lab-automated-essay-scoring-2/sample_submission.csv"
)
submission




,essay_id,score
0,d550b2d,4
1,0c10954,5
2,ef04816,3
3,88ab8d9,5
4,cee8c0f,5
...,...,...
1726,d8c9e3d,3
1727,6c3e2b9,5
1728,743f56b,5
1729,03630c7,1


In [7]:
cList = {
    "ain't": "am not",
    "aren't": "are not",
    "can't": "cannot",
    "can't've": "cannot have",
    "'cause": "because",
    "could've": "could have",
    "couldn't": "could not",
    "couldn't've": "could not have",
    "didn't": "did not",
    "doesn't": "does not",
    "don't": "do not",
    "hadn't": "had not",
    "hadn't've": "had not have",
    "hasn't": "has not",
    "haven't": "have not",
    "he'd": "he would",
    "he'd've": "he would have",
    "he'll": "he will",
    "he'll've": "he will have",
    "he's": "he is",
    "how'd": "how did",
    "how'd'y": "how do you",
    "how'll": "how will",
    "how's": "how is",
    "I'd": "I would",
    "I'd've": "I would have",
    "I'll": "I will",
    "I'll've": "I will have",
    "I'm": "I am",
    "I've": "I have",
    "isn't": "is not",
    "it'd": "it had",
    "it'd've": "it would have",
    "it'll": "it will",
    "it'll've": "it will have",
    "it's": "it is",
    "let's": "let us",
    "ma'am": "madam",
    "mayn't": "may not",
    "might've": "might have",
    "mightn't": "might not",
    "mightn't've": "might not have",
    "must've": "must have",
    "mustn't": "must not",
    "mustn't've": "must not have",
    "needn't": "need not",
    "needn't've": "need not have",
    "o'clock": "of the clock",
    "oughtn't": "ought not",
    "oughtn't've": "ought not have",
    "shan't": "shall not",
    "sha'n't": "shall not",
    "shan't've": "shall not have",
    "she'd": "she would",
    "she'd've": "she would have",
    "she'll": "she will",
    "she'll've": "she will have",
    "she's": "she is",
    "should've": "should have",
    "shouldn't": "should not",
    "shouldn't've": "should not have",
    "so've": "so have",
    "so's": "so is",
    "that'd": "that would",
    "that'd've": "that would have",
    "that's": "that is",
    "there'd": "there had",
    "there'd've": "there would have",
    "there's": "there is",
    "they'd": "they would",
    "they'd've": "they would have",
    "they'll": "they will",
    "they'll've": "they will have",
    "they're": "they are",
    "they've": "they have",
    "to've": "to have",
    "wasn't": "was not",
    "we'd": "we had",
    "we'd've": "we would have",
    "we'll": "we will",
    "we'll've": "we will have",
    "we're": "we are",
    "we've": "we have",
    "weren't": "were not",
    "what'll": "what will",
    "what'll've": "what will have",
    "what're": "what are",
    "what's": "what is",
    "what've": "what have",
    "when's": "when is",
    "when've": "when have",
    "where'd": "where did",
    "where's": "where is",
    "where've": "where have",
    "who'll": "who will",
    "who'll've": "who will have",
    "who's": "who is",
    "who've": "who have",
    "why's": "why is",
    "why've": "why have",
    "will've": "will have",
    "won't": "will not",
    "won't've": "will not have",
    "would've": "would have",
    "wouldn't": "would not",
    "wouldn't've": "would not have",
    "y'all": "you all",
    "y'alls": "you alls",
    "y'all'd": "you all would",
    "y'all'd've": "you all would have",
    "y'all're": "you all are",
    "y'all've": "you all have",
    "you'd": "you had",
    "you'd've": "you would have",
    "you'll": "you you will",
    "you'll've": "you you will have",
    "you're": "you are",
    "you've": "you have",
}




In [8]:
c_re = re.compile("(%s)" % "|".join(cList.keys()))




In [9]:
def expandContractions(text, c_re=c_re):
    def replace(match):
        return cList[match.group(0)]

    return c_re.sub(replace, text)




In [10]:
def removeHTML(x):
    html = re.compile(r"<.*?>")
    return html.sub(r"", x)




In [11]:
# Optimized preprocessing: replace the costly Series.apply with a vectorized str.replace call.
def dataPreprocessing(series):
    s = series.str.lower()
    s = s.str.replace(r"<.*?>", "", regex=True)  # remove HTML
    s = s.str.replace(r"@\w+", "", regex=True)  # remove mentions
    s = s.str.replace(r"'\d+", "", regex=True)  # remove apostrophe‑numbers
    s = s.str.replace(r"\d+", "", regex=True)  # remove digits
    s = s.str.replace(r"http\w+", "", regex=True)  # remove URLs
    s = s.str.replace(r"\s+", " ", regex=True)  # collapse whitespace
    # vectorized contraction expansion
    s = s.str.replace(c_re, lambda m: cList[m.group(0)], regex=True)
    s = s.str.replace(r"\.+", ".", regex=True)  # collapse repeated periods
    s = s.str.replace(r",+", ",", regex=True)  # collapse commas
    s = s.str.replace(r"\n", "", regex=True)  # remove newlines
    s = s.str.replace(r"[^\w\s]", "", regex=True)  # remove punctuation
    s = s.str.strip()
    return s




In [12]:
x = dataPreprocessing(train_df1["full_text"])




In [13]:
test_df1 = pd.read_csv(
    "/kaggle/input/learning-agency-lab-automated-essay-scoring-2/test.csv"
)




In [14]:
x0 = dataPreprocessing(test_df1["full_text"])




In [15]:
y = train_df1["score"]




In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.1, random_state=123, stratify=y
)




In [17]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)




(14018,)
(14018,)
(1558,)
(1558,)


In [18]:
text_vectorizer = TfidfVectorizer(
    stop_words=None,
    sublinear_tf=True,
    strip_accents="unicode",
    binary=False,
    analyzer="word",
    token_pattern=r"\w{1,}",
    ngram_range=(1, 3),  # now includes trigrams
    norm="l2",
    use_idf=True,
    smooth_idf=True,
    max_features=600000,  # larger vocabulary
    min_df=1,
)




In [19]:
X_train_features = text_vectorizer.fit_transform(X_train)




In [20]:
base_clf = LinearSVC(
    C=100.0,  # slightly stronger regularisation
    dual=False,
    random_state=123,
    tol=1e-5,
    class_weight="balanced",
)
clf = CalibratedClassifierCV(
    base_estimator=base_clf, cv=3, method="isotonic", n_jobs=-1
)
clf.fit(X_train_features, y_train)

test_features = text_vectorizer.transform(X_test)
probs = clf.predict_proba(test_features)
class_labels = clf.classes_
y_pred_cont = np.dot(probs, class_labels)
y_pred = np.rint(y_pred_cont).astype(int)
y_pred = np.clip(y_pred, 1, 6)




/usr/local/lib/python3.11/dist-packages/sklearn/calibration.py:321: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(


In [21]:
print(confusion_matrix(y_test.values, y_pred))




[[  5  94  12   1   0   0]
 [  0 177 240   8   0   0]
 [  0  45 441  77   0   0]
 [  0   3 142 211   0   0]
 [  0   0   3  83   2   0]
 [  0   0   0  12   2   0]]


In [22]:
print(classification_report(y_test.values, y_pred))




              precision    recall  f1-score   support

           1       1.00      0.04      0.09       112
           2       0.55      0.42      0.48       425
           3       0.53      0.78      0.63       563
           4       0.54      0.59      0.56       356
           5       0.50      0.02      0.04        88
           6       0.00      0.00      0.00        14

    accuracy                           0.54      1558
   macro avg       0.52      0.31      0.30      1558
weighted avg       0.56      0.54      0.49      1558



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [23]:
kappa = cohen_kappa_score(y_test.values, y_pred, weights="quadratic")
print("Cohen's kappa score:", kappa)




Cohen's kappa score: 0.6570270804609932


In [24]:
# Use the already‑trained vectorizer and calibrated model to predict the full test set,
# avoiding a second TF‑IDF fit and a second CalibratedClassifierCV training.
test_features_full = text_vectorizer.transform(x0)
test_probs_full = clf.predict_proba(test_features_full)
test_pred_cont_full = np.dot(test_probs_full, clf.classes_)
test_predictions_full = np.rint(test_pred_cont_full).astype(int)
test_predictions_full = np.clip(test_predictions_full, 1, 6)




In [25]:
submission = pd.read_csv(
    "/kaggle/input/learning-agency-lab-automated-essay-scoring-2/sample_submission.csv"
)
submission["score"] = test_predictions_full
submission.to_csv("submission.csv", index=False)
display(submission)

,essay_id,score
0,d550b2d,2
1,0c10954,2
2,ef04816,4
3,88ab8d9,2
4,cee8c0f,2
...,...,...
1726,d8c9e3d,4
1727,6c3e2b9,2
1728,743f56b,3
1729,03630c7,3
